In [ ]:
import h5py
import pickle

import pandas as pd
import numpy as np
import xarray as xr

%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patches as patches

import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
epi_threshold = 0.95 # for hw definition in EPI
pctl = 0.75 # which percentile of the BLH distribution to track

In [ ]:
# General definition of PRUDENCE regions:
prudence_regions = {"BI": {"name": "British Isles", "lon": slice(-10, 2), "lat": slice(50, 59), "label_pos": (0.5, 57.5), "color": "tab:blue"},
                    "IP": {"name": "Iberian Peninsula", "lon": slice(-10, 3), "lat": slice(36, 44), "label_pos": (-6, 40.0), "color": "tab:orange"},
                    "FR": {"name": "France", "lon": slice(-5, 5), "lat": slice(44, 50), "label_pos": (2.0, 46.0), "color": "tab:green"},
                    "ME": {"name": "Mid-Europe", "lon": slice(2, 16), "lat": slice(48, 55), "label_pos": (12.0, 52.), "color": "tab:red"},
                    "SC": {"name": "Scandinavia", "lon": slice(5, 30), "lat": slice(55, 70), "label_pos": (14.5, 62.5), "color": "tab:purple"},
                    "AL": {"name": "Alps", "lon": slice(5, 15), "lat": slice(44, 48), "label_pos": (11.0, 46.0), "color": "tab:brown"},
                    "MD": {"name": "Mediterranean", "lon": slice(3, 25), "lat": slice(36, 44), "label_pos": (18.5, 37.5), "color": "tab:pink"},
                    "EA": {"name": "Eastern Europe", "lon": slice(16, 30), "lat": slice(44, 55), "label_pos": (19.5, 52.), "color": "tab:grey"}}

In [ ]:
def find_events(hw_ind):
    """
    Finds start and end dates of heatwaves, i.e., three consecutive hot days and their length.
    The input xarray data array 'hw_ind' is boolean, where hot conditions equals True.
    """

    # Get values and datetimes:
    bools = hw_ind.values
    dts = hw_ind["time"].values
    
    # Switching from Flase to True yields 1 from np.diff and vice versa
    diff = np.diff(np.concatenate(([False], bools, [False])).astype(int))
    start_idxs = np.where(diff == 1)[0]
    end_idxs = np.where(diff == -1)[0] - 1
    
    events = []
    for start_idx, end_idx in zip(start_idxs, end_idxs):
        start_time = dts[start_idx].astype("datetime64[D]")
        end_time = dts[end_idx].astype("datetime64[D]")
        length = (end_idx - start_idx + 1).item()
        events.append((start_time, end_time, length))

    return events

# EPI Timeseries

## EPI Comparison of the Mid-European PRUDENCE Region

In [ ]:
epi_rea6 = xr.open_dataset("./data/EPI/REA6/ME/ExtremalPatternIndex_TPDM.nc")["EPI"]
q_epi_rea6 = np.quantile(epi_rea6, epi_threshold) # 95th percentile threshold for hot day definition
dts_rea6 = epi_rea6["time"].values.astype("datetime64[D]")

epi_era5 = xr.open_dataset("./data/EPI/ERA5/ME/ExtremalPatternIndex_TPDM.nc")["EPI"]
q_epi_era5 = np.quantile(epi_era5, epi_threshold)
dts_era5 = epi_era5["time"].values.astype("datetime64[D]")

In [ ]:
start = 2014
n = 5

fig, axs = plt.subplots(1, n, figsize=(12,4), sharey=True)
fig.subplots_adjust(wspace=0.1) 

for i, year in enumerate(range(start, start + n)):
    
    # Plot ERA5:
    mask_e = dts_era5.astype("datetime64[Y]") == np.datetime64(str(year))
    if i == n-1:
        axs[i].plot(dts_era5[mask_e], epi_era5[mask_e], label="ERA5", color="tab:blue")
    else:
        axs[i].plot(dts_era5[mask_e], epi_era5[mask_e], color="tab:blue")

    axs[i].hlines(q_epi_era5, 
                  np.datetime64(f"{year}-06-01"), 
                  min(np.datetime64(f"{year}-10-01"), np.datetime64(f"{start+n-1}-09-01")), 
                  clip_on=False, color="tab:blue", linestyle="dashed")
    
    # Plot REA6:
    mask_r = dts_rea6.astype("datetime64[Y]") == np.datetime64(str(year))
    if i == n-1:
        axs[i].plot(dts_rea6[mask_r], epi_rea6[mask_r], label="REA6", color="tab:red")
    else:
        axs[i].plot(dts_rea6[mask_r], epi_rea6[mask_r], color="tab:red")

    axs[i].hlines(q_epi_rea6, 
                  np.datetime64(f"{year}-06-01"), 
                  min(np.datetime64(f"{year}-10-01"), np.datetime64(f"{start+n-1}-09-01")), 
                  clip_on=False, color="tab:red", linestyle="dashed")
    

    # Cosmetics:
    axs[i].set_xlim(np.union1d(dts_era5[mask_e], dts_rea6[mask_r])[[0,-1]])
    axs[i].xaxis.set_tick_params(rotation=90)

    # Making the subplots appear as one:
    if i == 0:
        axs[i].spines.right.set_visible(False)
    elif i < n - 1:    
        axs[i].spines.right.set_visible(False)
        axs[i].spines.left.set_visible(False)
        axs[i].yaxis.set_tick_params(length=0.) #hack for making them invisible (setting ticks to [] removes all b/c sharey=True?)
    else:
        axs[i].spines.left.set_visible(False)
        axs[i].yaxis.set_tick_params(length=0.)
        axs[i].legend()
    axs[i].spines.top.set_visible(False)

axs[0].set_ylabel("EPI")


axs[1].scatter(np.datetime64("2015-07-03"), 0, marker="x", color="black", linewidth=2)
axs[1].scatter(np.datetime64("2015-08-07"), 0, marker="x", color="black", linewidth=2)

axs[2].scatter(np.datetime64("2016-08-26"), 0, marker="x", color="black", linewidth=2)

axs[4].scatter(np.datetime64("2018-07-27"), 0, marker="x", color="black", linewidth=2)
axs[4].scatter(np.datetime64("2018-08-08"), 0, marker="x", color="black", linewidth=2)

plt.savefig('./figs/epi_comp.pdf', bbox_inches='tight', format='pdf')
plt.show()

## Calculate Numbers of Hot Days and Heatwaves

In [ ]:
def find_block_lengths(ts):
    """
    Takes a boolean time series 'ts' and finds consecutive hot days.
    """

    # Fill single-False gaps if desired:
    filled = ts[:]
    #for i in range(1, len(filled) - 1):
    #    if not filled[i] and filled[i-1] and filled[i+1]:
    #        filled[i] = True
    
    # Find consecutive True blocks:
    lengths = []
    current_length = 0
    
    for val in filled:
        if val:
            current_length += 1
        else:
            if current_length > 0:
                lengths.append(current_length)
                current_length = 0
    
    if current_length > 0:
        lengths.append(current_length)
    
    return np.array(lengths)

Here, Heatwaves are three consecutive hot days without gaps.

In [ ]:
for key in prudence_regions.keys():
    # Load EPI:
    epi_rea6 = xr.open_dataset(f"./data/EPI/REA6/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    q_epi_rea6 = np.quantile(epi_rea6, epi_threshold) # 95th percentile threshold for hot day definition
    dts_rea6 = epi_rea6["time"].values.astype("datetime64[D]")

    epi_era5 = xr.open_dataset(f"./data/EPI/ERA5/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    q_epi_era5 = np.quantile(epi_era5, epi_threshold)
    dts_era5 = epi_era5["time"].values.astype("datetime64[D]")

    # Get events:
    lengths_rea6_14to18 = find_block_lengths(list(epi_rea6.sel(time=slice(np.datetime64("2014"), np.datetime64("2019"))) > q_epi_rea6))
    lengths_era5_14to18 = find_block_lengths(list(epi_era5.sel(time=slice(np.datetime64("2014"), np.datetime64("2019"))) > q_epi_era5))
    lengths_era5_00to22 = find_block_lengths(list(epi_era5.sel(time=slice(np.datetime64("2000"), np.datetime64("2023"))) > q_epi_era5))

    print(prudence_regions[key]["name"])
    print(f"REA6 (2014 - 2018): {sum(lengths_rea6_14to18)} hot days with {(lengths_rea6_14to18 > 2).sum()} heatwaves.")
    print(f"ERA5 (2014 - 2018): {sum(lengths_era5_14to18)} hot days with {(lengths_era5_14to18 > 2).sum()} heatwaves.")
    print(f"ERA5 (2000 - 2022): {sum(lengths_era5_00to22)} hot days with {(lengths_era5_00to22 > 2).sum()} heatwaves.")
    print()

# Results of Grid Point Tests

In [ ]:
from scipy.stats import mannwhitneyu, PermutationMethod

In [ ]:
def holm(alpha, p_vals):
    """
    Computes the p-value threshold for a family wise error rate of alpha
    using Holm's method.

    Holm, Sture. "A simple sequentially rejective multiple test procedure." Scandinavian journal of statistics (1979): 65-70.    
    """

    p = p_vals.reshape(-1)
    p = p[~np.isnan(p)] 
    p = sorted(p)

    m = len(p)

    # Step-up:
    for k in range(m):
        if p[k] <= alpha / (m - k):
            continue
        else:
            return k/m

## Data

### REA6

Note that the REA6 domain is cut at the Western and Southern edges. This was done to lessen the computational burden. The Western border is mostly sea. The Southern part of the domain contains North Africa, where the PBL is quite different to Europe.

In [ ]:
# Grid info:
rea_static = xr.open_dataset("./data/rea_static_rot.nc")
rea_lsm = rea_static["LSM"].coarsen(dim={'rlat': 3, 'rlon': 3}, boundary='trim').max(skipna=True)
rea_lsm = rea_lsm.isel(rlat=slice(49,None), rlon=slice(29,None))

# Definition of hot days/heatwaves:
hw_rea6 = xr.open_dataset("./data/hw_gridded_rea6.nc").isel(rlat=slice(49,None),rlon=slice(29,None))["hw_ind"]

# Testing results using VT-BLH:
with h5py.File("./data/emd_rea6_vt_blh.hdf5", "r") as f:
    p_rea6_vt = f["p"][()]
p_rea6_vt[rea_lsm < 0.5] = np.nan #cut out with LSM

# Testing results using IFS-BLH:
with h5py.File("./data/emd_rea6_ifs_blh.hdf5", "r") as f: 
    p_rea6_ifs = f["p"][()]
p_rea6_ifs[rea_lsm < 0.5] = np.nan

### ERA5

In [ ]:
# Definition of hot days/heatwaves:
hw_era5 = xr.open_dataset("./data/hw_gridded_era5.nc")["hw_ind"]

# Grid info:
era_lsm = xr.open_dataset("data/era5_lsm.nc").squeeze()["lsm"]
era_lsm["longitude"] = np.where(era_lsm["longitude"] > 180., era_lsm["longitude"] - 360., era_lsm["longitude"])
era_lsm = era_lsm.sel(latitude=hw_era5["latitude"], longitude=hw_era5["longitude"])

# Short and long period testing results (IFS-BLH):
with h5py.File("./data/emd_era5_14to18.hdf5", "r") as f:
    p_era5_short = f["p"][()]
p_era5_short[era_lsm < 0.5] = np.nan 

with h5py.File("./data/emd_era5_long.hdf5", "r") as f:
    p_era5_long = f["p"][()]
p_era5_long[era_lsm < 0.5] = np.nan 

### Observations

In [ ]:
_ = [pd.read_csv(f"./data/obs/blhs_era5_obs_ifs_{YYYY}.csv", index_col=0, parse_dates=["DateTime"]) for YYYY in range(2010,2023)]
df_obs = pd.concat(_, ignore_index=True)
df_obs = df_obs[df_obs.ID != 0.] #for some reason ID 0 does weird things

# Filter low res BUFR:
#df_obs = df_obs[df_obs["codetype"] == 109]

id_counts = df_obs.value_counts(["ID"]).reset_index(name="count") 
lons = np.array([np.mean(df_obs[df_obs["ID"] == id]["lon"], axis=0) for id in id_counts["ID"]])
lats = np.array([np.mean(df_obs[df_obs["ID"] == id]["lat"], axis=0) for id in id_counts["ID"]])

In [ ]:
df_results = pd.DataFrame({"ID": id_counts["ID"], 
                           "lat": lats, "lon": lons, 
                           "n_hw": np.nan, "n_nc": np.nan, 
                           "BLH_avg_hw": np.nan, "BLH_avg_nc": np.nan, 
                           "p": np.nan})

for idx, row in df_results.iterrows():
    df_all_day = df_obs[df_obs["ID"] == row["ID"]].copy()
    df_station = df_all_day[df_all_day["DateTime"].dt.hour == 12].copy() # only 12UTC
    
    hw_ind_station = hw_era5.sel(latitude=row["lat"], longitude=row["lon"], method="nearest")

    df_station["Date"] = df_station["DateTime"].dt.date
    hw_dates = pd.to_datetime(hw_ind_station["time"][hw_ind_station > 0.5]).normalize().date
    
    if (df_station["Date"].isin(hw_dates)).sum() < 5:
        continue
    else:
        u, p_val = mannwhitneyu(x=df_station[~df_station["Date"].isin(hw_dates)]["BLH"], 
                                y=df_station[df_station["Date"].isin(hw_dates)]["BLH"], 
                                use_continuity=False, alternative="less", method=PermutationMethod(n_resamples=10000, batch=100))
        df_results.loc[idx,"p"] = p_val

    df_results.loc[idx, "n_hw"] = (df_station["Date"].isin(hw_dates)).sum()
    df_results.loc[idx, "n_nc"] = (~df_station["Date"].isin(hw_dates)).sum()

    df_results.loc[idx, "BLH_avg_hw"] = np.mean(df_station[df_station["Date"].isin(hw_dates)]["BLH"])
    df_results.loc[idx, "BLH_avg_nc"] = np.mean(df_station[~df_station["Date"].isin(hw_dates)]["BLH"])

df_results["diff"] = df_results["BLH_avg_hw"] - df_results["BLH_avg_nc"]

### Figure

Cross hatching using the holm thresholds

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(10,4), sharex=True, sharey=True, subplot_kw={'projection': ccrs.PlateCarree()})
fig.tight_layout()
lvls = np.arange(0,1.1,0.1)

for i, title, label in zip(range(3), ["COSMO-REA6", "ERA5", "12 UTC Observations"], ["a)", "b)", "c)"]):
    axs[i].set_extent([-15, 38, 35.5, 71.5], crs=ccrs.PlateCarree())

    axs[i].add_feature(cfeature.OCEAN, zorder=1)
    axs[i].add_feature(cfeature.COASTLINE, zorder=20)
    axs[i].add_feature(cfeature.LAKES, zorder=20)

    axs[i].set(title=title)

    axs[i].text(0.05, 0.93, label, transform = axs[i].transAxes, ha="center", va="center", fontweight="bold", 
            zorder=5, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))


# COSMO-REA6 (IFS)
ax = axs[0]
gl1 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, zorder=10, color="lightgrey")
gl1.top_labels = False
gl1.right_labels = False

im = ax.contourf(hw_rea6["lon"], hw_rea6["lat"], p_rea6_ifs, levels=lvls, cmap="viridis_r", zorder=15)
ax.pcolor(hw_rea6["lon"], hw_rea6["lat"], np.ma.masked_greater(p_rea6_ifs, 0.01), hatch='///', alpha=0., zorder=16)

p_threshold_holm_rea6 = np.nanquantile(p_rea6_ifs, holm(0.1, p_rea6_ifs))
ax.pcolor(hw_rea6["lon"], hw_rea6["lat"], np.ma.masked_greater(p_rea6_ifs, p_threshold_holm_rea6), hatch='\\\\\\', alpha=0., zorder=17)


# ERA5
ax = axs[1]
gl2 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, zorder=10, color="lightgrey")
gl2.top_labels = False
gl2.right_labels = False
gl2.left_labels = False

im = ax.contourf(hw_era5["longitude"], hw_era5["latitude"], p_era5_short, levels=lvls, cmap="viridis_r", zorder=15)
ax.pcolor(hw_era5["longitude"], hw_era5["latitude"], np.ma.masked_greater(p_era5_short, 0.01), hatch='///', alpha=0., zorder=16)

p_threshold_holm_era5 = np.nanquantile(p_era5_short, holm(0.1, p_era5_short))
ax.pcolor(hw_era5["longitude"], hw_era5["latitude"], np.ma.masked_greater(p_era5_short, p_threshold_holm_era5), hatch='\\\\\\', alpha=0., zorder=17)


# 12 UTC Observations
ax = axs[2]
gl3 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, zorder=10, color="lightgrey")
gl3.top_labels = False
gl3.right_labels = False
gl3.left_labels = False

cmap = plt.cm.viridis_r
bounds = np.arange(0,1.1,0.1)
norm = mpl.colors.BoundaryNorm(bounds, cmap.N)

scatter = ax.scatter(df_results["lon"], df_results["lat"], c=df_results["p"],
    cmap=cmap, norm=norm, s=60, edgecolor="k", zorder=20)

# Make room for colorbar:
fig.subplots_adjust(right=0.9)
cbar_ax = fig.add_axes([0.92, 0.2, 0.02, 0.6])  # [left, bottom, width, height]
fig.colorbar(im, cax=cbar_ax, label="p-Value")


plt.savefig("./figs/p_vals_dmax_comp_holm.png", bbox_inches="tight", format="png", dpi=600)
plt.show()

### Values

In [ ]:
alpha = 0.01 # single cell significance threshold

# REA6 VT-BLH:
finite_rea6_vt = p_rea6_vt[np.isfinite(p_rea6_vt)]
p_threshold_holm_rea6_vt = np.nanquantile(p_rea6_vt, holm(0.1, p_rea6_vt))

print(f"REA6 (VT) p-values below {alpha}:\t\t{np.sum(finite_rea6_vt.flatten() < alpha)/len(finite_rea6_vt) * 100:.1f}%"
      + f"\t Holm:\t{np.sum(finite_rea6_vt.flatten() < p_threshold_holm_rea6_vt)/len(finite_rea6_vt) * 100:.1f}%")

# REA6 IFS-BLH:
finite_rea6_ifs = p_rea6_ifs[np.isfinite(p_rea6_ifs)]
p_threshold_holm_rea6_ifs = np.nanquantile(p_rea6_ifs, holm(0.1, p_rea6_ifs))

print(f"REA6 (IFS) p-values below {alpha}:\t\t{np.sum(finite_rea6_ifs.flatten() < alpha)/len(finite_rea6_ifs) * 100:.1f}%"
      + f"\t Holm:\t{np.sum(finite_rea6_ifs.flatten() < p_threshold_holm_rea6_ifs)/len(finite_rea6_ifs) * 100:.1f}%")

# ERA5 short:
finite_era5_short = p_era5_short[np.isfinite(p_era5_short)]
p_threshold_holm_era5_short = np.nanquantile(p_era5_short, holm(0.1, p_era5_short))

print(f"ERA5 (short) p-values below {alpha}:\t{np.sum(finite_era5_short.flatten() < alpha)/len(finite_era5_short) * 100:.1f}%"
      + f"\t Holm:\t{np.sum(finite_era5_short.flatten() < p_threshold_holm_era5_short)/len(finite_era5_short) * 100:.1f}%")

# ERA5 long:
finite_era5_long = p_era5_long[np.isfinite(p_era5_long)]
p_threshold_holm_era5_long = np.nanquantile(p_era5_long, holm(0.1, p_era5_long))

print(f"ERA5 (long) p-values below {alpha}:\t{np.sum(finite_era5_long.flatten() < alpha)/len(finite_era5_long) * 100:.1f}%" 
      + f"\t Holm:\t{np.sum(finite_era5_long.flatten() < p_threshold_holm_era5_long)/len(finite_era5_long) * 100:.1f}%")

# BLH Q-Q Plots

Functions for estimating the confindence bands

In [ ]:
def pseudotimeseries(timeseries, sample_size, p):
    """
    Generate a pseudo time series by resampling `timeseries` using the
    stationary bootstrap method:
    [1] Politis, Dimitris N., and Joseph P. Romano. "The stationary bootstrap." 
    Journal of the American Statistical association 89.428 (1994): 1303-1313.
    
    Parameters:
    - timeseries: numpy array
    - sample_size: length of generated series (int)
    - p: geometric distribution parameter controlling block length
    
    Returns:
    - pseudo_ts: resampled time series of same length
    """
    n = 0  # current length of pseudo sample
    N = len(timeseries)

    # Initialize output
    pseudo_ts = np.empty(sample_size, dtype=timeseries.dtype)
    
    while n < sample_size:
        I = np.random.randint(0, N) # Block starting point
        L = np.random.geometric(p) # Block length
        
        # Prevent overflow in the last assignment
        if n + L > sample_size:
            L = sample_size - n
        
        # Cyclic boundary condition
        if I + L <= N:
            pseudo_ts[n:n+L] = timeseries[I:I+L]
        else:
            # Wrap around using modulo
            indices = np.arange(I, I+L) % N
            pseudo_ts[n:n+L] = timeseries[indices]
        
        n += L
    
    return pseudo_ts

def bootstrap_qq_conf(qs, blh, nc_days, hw_days, n_boot, alpha):
    """
    Estimates uncertainty of the normal condition quantiles by bootstrapping.
    When resampling, as many days are drawn as there are heat wave days, because
    this represents the uncertainity if there was no difference between 'hot and normal'.
    We use the stationary bootstrap.
    """
    n_hw = len(hw_days)

    qs_resamples = np.full((n_boot, len(qs)), np.nan)

    for i in range(n_boot):
        t_resample = pseudotimeseries(nc_days, n_hw, 0.3) # Normal bootstrap, replace with: np.random.choice(nc_days, n_hw, replace=True)
        qs_resamples[i] = np.nanquantile(blh.sel(time=t_resample), qs)

    lower = np.percentile(qs_resamples, 100 * alpha/2, axis=0)
    upper = np.percentile(qs_resamples, 100 * (1-alpha/2), axis=0)

    return lower, upper

Shared data

In [ ]:
# Grid info:
rea_static = xr.open_dataset("./data/rea_static_rot.nc")
rea6_lsm = rea_static["LSM"].coarsen(dim={'rlat': 3, 'rlon': 3}, boundary='trim').max(skipna=True)
rea6_lsm = rea6_lsm.isel(rlat=slice(49,None), rlon=slice(29,None))

era5_lsm = xr.open_dataset("./data/era5_lsm.nc").squeeze()["lsm"]
era5_lsm["longitude"] = xr.where(era5_lsm["longitude"] > 180., era5_lsm["longitude"] - 360, era5_lsm["longitude"])


# BLHs:
blh_rea6_vt = xr.open_dataset("./data/vt_blh_dmax_rea6.nc")["blh"].isel(rlat=slice(49,None),rlon=slice(29,None)).where(rea6_lsm > 0.5)
blh_rea6_ifs = xr.open_dataset("./data/ifs_blh_dmax_rea6.nc")["blh"].isel(rlat=slice(49,None),rlon=slice(29,None)).where(rea6_lsm > 0.5)

blh_era5 = xr.open_dataset("./data/blh_dmax_era5.nc")["blh"].where(era5_lsm > 0.5)
blh_era5["valid_time"] = blh_era5["valid_time"].dt.floor("D")

## Global

In [ ]:
# Definition of hot days/heatwaves:
hw_era5 = xr.open_dataset("./data/hw_gridded_era5.nc")["hw_ind"]
hw_era5["time"] = hw_era5["time"].dt.floor("D")
hw_era5 = hw_era5.rename({"time": "valid_time"})

hw_rea6 = xr.open_dataset("./data/hw_gridded_rea6.nc").isel(rlat=slice(49,None),rlon=slice(29,None))["hw_ind"]

In [ ]:
qs = np.arange(0.05, 1.0, 0.05)

# Uncertainty estimation:
alpha = 0.05 # confidence band of 1 - alpha
n_boot = 1000

fig, axs = plt.subplots(1, 2, sharex=True, sharey=True, tight_layout=True, figsize=(7,4))

# REA6 - IFS
blh_nc = blh_rea6_ifs.where(np.isnan(hw_rea6)).data.reshape(-1)
quantiles_nc = np.nanquantile(blh_nc, qs) / 1000

blh_hw = blh_rea6_ifs.where(np.isfinite(hw_rea6)).data.reshape(-1)
quantiles_hw = np.nanquantile(blh_hw, qs) / 1000

# Confidence bands:
lower, upper = bootstrap_qq_conf(qs, blh_rea6_ifs, blh_rea6_ifs["time"], np.zeros(30), n_boot, alpha)
lower, upper = lower / 1000, upper / 1000


axs[0].scatter(quantiles_nc, quantiles_hw, color="tab:blue")
axs[0].scatter(quantiles_nc[[4,9,14]], quantiles_hw[[4,9,14]], color="white", edgecolors="tab:blue")
axs[0].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
axs[0].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)



axs[0].plot([0, 4], [0, 4], color="black", zorder=-2)
axs[0].set(title="COSMO-REA6")
axs[0].set_xticks([0, 1, 2, 3, 4])
axs[0].set_yticks([0, 1, 2, 3, 4])
axs[0].grid()
axs[0].text(0.07, 3.8, "a)", 
        ha='center', va='center', fontweight='bold', zorder=5,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
axs[0].set_aspect("equal")


# ERA5
era_lsm = xr.open_dataset("data/era5_lsm.nc").squeeze()["lsm"]
era_lsm["longitude"] = np.where(era_lsm["longitude"] > 180., era_lsm["longitude"] - 360., era_lsm["longitude"])
era_lsm = era_lsm.sel(latitude=hw_era5["latitude"], longitude=hw_era5["longitude"])

blh_nc = blh_era5.where(np.isnan(hw_era5)).data.reshape(-1)
quantiles_nc = np.nanquantile(blh_nc, qs) / 1000

blh_hw = blh_era5.where(np.isfinite(hw_era5)).data.reshape(-1)
quantiles_hw = np.nanquantile(blh_hw, qs) / 1000

# Confidence bands:
lower, upper = bootstrap_qq_conf(qs, blh_era5.rename({"valid_time": "time"}), blh_era5["valid_time"], np.zeros(30), n_boot, alpha)
lower, upper = lower / 1000, upper / 1000


axs[1].scatter(quantiles_nc, quantiles_hw)
axs[1].scatter(quantiles_nc[[4,9,14]], quantiles_hw[[4,9,14]], color="white", edgecolors="tab:blue")
axs[1].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
axs[1].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)


axs[1].plot([0, 4], [0, 4], color="black", zorder=-2)
axs[1].set(title="ERA5")
axs[1].set_xticks([0, 1, 2, 3, 4])
axs[1].set_yticks([0, 1, 2, 3,4 ])
axs[1].grid()
axs[1].text(0.07, 3.8, "b)", 
        ha='center', va='center', fontweight='bold', zorder=5,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))
axs[1].set_aspect("equal")

fig.supxlabel("Normal condition BLH quantiles in km")
fig.supylabel("Hot day BLH quantiles in km")
plt.savefig('./figs/global_comp.pdf', bbox_inches='tight', format='pdf')
plt.show()

## PRUDENCE Regions

### Reanalyses Comparison

In [ ]:
qs = np.arange(0.05, 1.0, 0.05)

# Uncertainty estimation:
alpha = 0.05 # confidence band of 1 - alpha
n_boot = 1000

fig, ax = plt.subplots(2, 4, figsize=(10,5.5), sharex=True, sharey=True, tight_layout=True)

for i, key, char in zip(range(8), prudence_regions.keys(), ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"]):
    j, k = np.unravel_index(i, (2,4))

    # Load HW definition for region:
    EPI = xr.open_dataset(f"./data/EPI/REA6/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"].sel(time=slice(np.datetime64("2014"), np.datetime64("2019")))
    EPI["time"] = EPI["time"].dt.floor("D")
    q_epi_region = np.quantile(EPI, epi_threshold)

    # REA6:
    mask_lat = (blh_rea6_ifs["lat"] >= (prudence_regions[key]["lat"]).start) & (blh_rea6_ifs["lat"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_rea6_ifs["lon"] >= (prudence_regions[key]["lon"]).start) & (blh_rea6_ifs["lon"] <= (prudence_regions[key]["lon"]).stop)
    blh_rea6_region = blh_rea6_ifs.where(mask_lat & mask_lon)

    hw_days = np.intersect1d(EPI["time"].where(EPI > q_epi_region, drop=True).values.astype("datetime64[D]"), blh_rea6_ifs["time"].values)
    blh_rea6_hw = blh_rea6_region.sel(time=hw_days)
    quantiles_hw = np.nanquantile(blh_rea6_hw, qs) / 1000
    
    nc_days = np.intersect1d(EPI["time"].where(EPI < q_epi_region, drop=True).values.astype("datetime64[D]"), blh_rea6_ifs["time"].values)
    blh_rea6_nc = blh_rea6_region.drop_sel(time=hw_days)
    quantiles_nc = np.nanquantile(blh_rea6_nc, qs) / 1000

    # Confidence band:
    lower, upper = bootstrap_qq_conf(qs, blh_rea6_region, nc_days, hw_days, n_boot, alpha)
    lower, upper = lower / 1000, upper / 1000
    
    if i == 7:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:blue", label="REA6")
        ax[j,k].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, 4], [0, 4], color="black")
    else:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:blue")
        ax[j,k].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, 4], [0, 4], color="black")

    
    # Same for ERA5:
    EPI = xr.open_dataset(f"./data/EPI/ERA5/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    EPI = EPI.sel(time=slice(np.datetime64("2014"), np.datetime64("2019"))) # select same time period as COSMO-REA6
    q_epi_region = np.quantile(EPI, epi_threshold)
    
    mask_lat = (blh_era5["latitude"] >= (prudence_regions[key]["lat"]).start) & (blh_era5["latitude"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_era5["longitude"] >= (prudence_regions[key]["lon"]).start) & (blh_era5["longitude"] <= (prudence_regions[key]["lon"]).stop)
    blh_era5_region = blh_era5.where(mask_lat & mask_lon)

    hw_days = np.intersect1d(EPI["time"].where(EPI > q_epi_region, drop=True).values.astype("datetime64[D]"), blh_era5["valid_time"].values)
    blh_era5_hw = blh_era5_region.sel(valid_time=hw_days)
    quantiles_hw = np.nanquantile(blh_era5_hw, qs) / 1000

    nc_days = np.intersect1d(EPI["time"].where(EPI < q_epi_region, drop=True).values.astype("datetime64[D]"), blh_era5["valid_time"].values)
    blh_era5_nc = blh_era5_region.drop_sel(valid_time=hw_days)
    quantiles_nc = np.nanquantile(blh_era5_nc, qs) / 1000

    # Confidence band:
    lower, upper = bootstrap_qq_conf(qs, blh_era5_region.rename({"valid_time": "time"}), nc_days, hw_days, n_boot, alpha)
    lower, upper = lower / 1000, upper / 1000

    if i == 7:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:red", label="ERA5")
        ax[j,k].plot(quantiles_nc, lower, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, 4.5], [0, 4.5], color="black")
    else:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:red")
        ax[j,k].plot(quantiles_nc, lower, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, 4.5], [0, 4.5], color="black")

    ax[j,k].set(title=prudence_regions[key]["name"])
    ax[j,k].set_xticks([0, 1, 2, 3, 4])
    ax[j,k].set_yticks([0, 1, 2, 3, 4])
    ax[j,k].set_aspect("equal")
    ax[j,k].grid()
    
    if i == 7:
        plt.legend(loc="lower right")

    ax[j,k].text(0.08, 0.92, char, transform = ax[j,k].transAxes, ha="center", va="center", fontweight="bold", 
            zorder=5, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

fig.supxlabel("Normal condition BLH quantiles in km")
fig.supylabel("Hot day BLH quantiles in km")

plt.savefig('./figs/prudence_qq_14to18_ifs.pdf', bbox_inches='tight', format='pdf')
plt.show()

In [ ]:
qs = np.arange(0.05, 1.0, 0.05)

# Uncertainty estimation:
alpha = 0.05 # confidence band of 1 - alpha
n_boot = 1000

fig, ax = plt.subplots(2, 4, figsize=(10,5.5), tight_layout=True)

for i, key, char, up_lim in zip(range(8), prudence_regions.keys(), ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"], [3, 4.5, 3, 3, 3, 3, 4.5, 3]):
    j, k = np.unravel_index(i, (2,4))

    # Load HW definition for region:
    EPI = xr.open_dataset(f"./data/EPI/REA6/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"].sel(time=slice(np.datetime64("2014"), np.datetime64("2019")))
    EPI["time"] = EPI["time"].dt.floor("D")
    q_epi_region = np.quantile(EPI, epi_threshold)

    # REA6:
    mask_lat = (blh_rea6_ifs["lat"] >= (prudence_regions[key]["lat"]).start) & (blh_rea6_ifs["lat"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_rea6_ifs["lon"] >= (prudence_regions[key]["lon"]).start) & (blh_rea6_ifs["lon"] <= (prudence_regions[key]["lon"]).stop)
    blh_rea6_region = blh_rea6_ifs.where(mask_lat & mask_lon)

    hw_days = np.intersect1d(EPI["time"].where(EPI > q_epi_region, drop=True).values.astype("datetime64[D]"), blh_rea6_ifs["time"].values)
    blh_rea6_hw = blh_rea6_region.sel(time=hw_days)
    quantiles_hw = np.nanquantile(blh_rea6_hw, qs) / 1000
    
    nc_days = np.intersect1d(EPI["time"].where(EPI < q_epi_region, drop=True).values.astype("datetime64[D]"), blh_rea6_ifs["time"].values)
    blh_rea6_nc = blh_rea6_region.drop_sel(time=hw_days)
    quantiles_nc = np.nanquantile(blh_rea6_nc, qs) / 1000

    # Confidence band:
    lower, upper = bootstrap_qq_conf(qs, blh_rea6_region, nc_days, hw_days, n_boot, alpha)
    lower, upper = lower / 1000, upper / 1000
    
    if i == 7:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:blue", label="REA6")
        ax[j,k].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, up_lim], [0, up_lim], color="black")
    else:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:blue")
        ax[j,k].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, up_lim], [0, up_lim], color="black")

    
    # Same for ERA5:
    EPI = xr.open_dataset(f"./data/EPI/ERA5/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    EPI = EPI.sel(time=slice(np.datetime64("2014"), np.datetime64("2019"))) # select same time period as COSMO-REA6
    q_epi_region = np.quantile(EPI, epi_threshold)
    
    mask_lat = (blh_era5["latitude"] >= (prudence_regions[key]["lat"]).start) & (blh_era5["latitude"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_era5["longitude"] >= (prudence_regions[key]["lon"]).start) & (blh_era5["longitude"] <= (prudence_regions[key]["lon"]).stop)
    blh_era5_region = blh_era5.where(mask_lat & mask_lon)

    hw_days = np.intersect1d(EPI["time"].where(EPI > q_epi_region, drop=True).values.astype("datetime64[D]"), blh_era5["valid_time"].values)
    blh_era5_hw = blh_era5_region.sel(valid_time=hw_days)
    quantiles_hw = np.nanquantile(blh_era5_hw, qs) / 1000

    nc_days = np.intersect1d(EPI["time"].where(EPI < q_epi_region, drop=True).values.astype("datetime64[D]"), blh_era5["valid_time"].values)
    blh_era5_nc = blh_era5_region.drop_sel(valid_time=hw_days)
    quantiles_nc = np.nanquantile(blh_era5_nc, qs) / 1000

    # Confidence band:
    lower, upper = bootstrap_qq_conf(qs, blh_era5_region.rename({"valid_time": "time"}), nc_days, hw_days, n_boot, alpha)
    lower, upper = lower / 1000, upper / 1000

    if i == 7:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:red", label="ERA5")
        ax[j,k].plot(quantiles_nc, lower, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, up_lim], [0, up_lim], color="black")
    else:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:red")
        ax[j,k].plot(quantiles_nc, lower, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, up_lim], [0, up_lim], color="black")

    ax[j,k].set(title=prudence_regions[key]["name"], xlim=(0, up_lim+0.3), ylim=(0, up_lim+0.3))
    ax[j,k].set_xticks(np.arange(0, max(up_lim,3.1), 1))#[0, 1, 2, 3, 4])
    ax[j,k].set_yticks(np.arange(0, max(up_lim,3.1), 1))#[0, 1, 2, 3, 4])
    ax[j,k].set_aspect("equal")
    ax[j,k].grid()
    
    if i == 7:
        plt.legend(loc="lower right")

    ax[j,k].text(0.08, 0.92, char, transform = ax[j,k].transAxes, ha="center", va="center", fontweight="bold", 
            zorder=5, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

fig.supxlabel("Normal condition BLH quantiles in km")
fig.supylabel("Hot day BLH quantiles in km")

plt.savefig('./figs/prudence_qq_14to18_ifs.pdf', bbox_inches='tight', format='pdf')
plt.show()

### ERA5 vs Observations

In [ ]:
# Observations:
_ = [pd.read_csv(f"./data/obs/blhs_era5_obs_ifs_{YYYY}.csv", index_col=0, parse_dates=["DateTime"]) for YYYY in range(2010,2023)]
df_obs_era5 = pd.concat(_).reset_index(drop=True)

# ERA5 model equivalents:
_ = [pd.read_csv(f"./data/obs/blhs_era5_veri0_ifs_{YYYY}.csv", index_col=0, parse_dates=["DateTime"]) for YYYY in range(2010,2023)]
df_me_era5 = pd.concat(_).reset_index(drop=True)

In [ ]:
fig, ax = plt.subplots(2,4,figsize=(10,5.5), sharex=True, sharey=True, tight_layout=True)

qs = np.arange(0.05, 1.0, 0.05)

# Uncertainty estimation:
alpha = 0.05 # confidence band of 1 - alpha
n_boot = 1000

for i, key, char in zip(range(8), prudence_regions.keys(), ["a)", "b)", "c)", "d)", "e)", "f)", "g)", "h)"]):
    j, k = np.unravel_index(i, (2,4))

    # Load HW definition for region:
    EPI = xr.open_dataset(f"./data/EPI/ERA5/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    q_epi_region = np.quantile(EPI, epi_threshold)
    hw_days = np.intersect1d(EPI["time"].where(EPI > q_epi_region, drop=True).values.astype("datetime64[D]"), blh_era5["valid_time"].values)

    # Radio Sondes:
    mask_lon = (df_obs_era5["lon"] >= prudence_regions[key]["lon"].start) & (df_obs_era5["lon"] <= prudence_regions[key]["lon"].stop)
    mask_lat = (df_obs_era5["lat"] >= prudence_regions[key]["lat"].start) & (df_obs_era5["lat"] <= prudence_regions[key]["lat"].stop)
    mask_day = df_obs_era5["DateTime"].dt.hour == 12 
    df_obs_region = df_obs_era5[mask_lon & mask_lat & mask_day]

    hw_mask = df_obs_region["DateTime"].dt.date.isin(hw_days.astype("datetime64[D]"))
    blh_obs_hw = df_obs_region[hw_mask]["BLH"]
    quantiles_hw = np.nanquantile(blh_obs_hw, qs) / 1000

    blh_obs_nc = df_obs_region[~hw_mask]["BLH"]
    quantiles_nc = np.nanquantile(blh_obs_nc, qs) / 1000


    # Confidence bands:
    # This is a simplified version of the function "bootstrap_qq_conf" as 
    # the station data is sparse and has gaps in the time series.
    resample = np.random.choice(blh_obs_nc, (n_boot, len(hw_days)), replace=True)
    qs_resample = np.quantile(resample, qs, axis=1)
    lower, upper = np.percentile(qs_resample, 100 * alpha/2, axis=1), np.percentile(qs_resample, 100 * (1-alpha/2), axis=1)
    lower, upper = lower / 1000, upper / 1000

    if i == 7:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:blue", label="Obs")
        ax[j,k].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, 3], [0, 3], color="black")
    else:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:blue")
        ax[j,k].plot(quantiles_nc, lower, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:blue", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot([0, 3], [0, 3], color="black")
    

    # ERA5:
    mask_lon = (df_me_era5["lon"] >= prudence_regions[key]["lon"].start) & (df_me_era5["lon"] <= prudence_regions[key]["lon"].stop)
    mask_lat = (df_me_era5["lat"] >= prudence_regions[key]["lat"].start) & (df_me_era5["lat"] <= prudence_regions[key]["lat"].stop)
    mask_day = df_me_era5["DateTime"].dt.hour == 12
    df_me_region = df_me_era5[mask_lon & mask_lat & mask_day]


    hw_mask = df_me_region["DateTime"].dt.date.isin(hw_days.astype("datetime64[D]"))
    blh_era5_hw = df_me_region[hw_mask]["BLH"]
    quantiles_hw = np.nanquantile(blh_era5_hw, qs) / 1000

    blh_era5_nc = df_me_region[~hw_mask]["BLH"]
    quantiles_nc = np.nanquantile(blh_era5_nc, qs) / 1000


    # Confidence bands:
    # This is a simplified version of the function "bootstrap_qq_conf" as 
    # the station data is sparse and has gaps in the time series.
    resample = np.random.choice(blh_era5_nc, (n_boot, len(hw_days)), replace=True)
    qs_resample = np.quantile(resample, qs, axis=1)
    lower, upper = np.percentile(qs_resample, 100 * alpha/2, axis=1), np.percentile(qs_resample, 100 * (1-alpha/2), axis=1)
    lower, upper = lower / 1000, upper / 1000

    if i == 7:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:red", label="ERA5")
        ax[j,k].plot(quantiles_nc, lower, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
    else:
        ax[j,k].scatter(quantiles_nc, quantiles_hw, marker=".", color="tab:red")
        ax[j,k].plot(quantiles_nc, lower, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)
        ax[j,k].plot(quantiles_nc, upper, color="tab:red", linestyle="dashed", alpha=0.8, zorder=-1)

    ax[j,k].set(title=prudence_regions[key]["name"], xlim=(0, 3.3), ylim=(0, 3.3))
    ax[j,k].set_xticks([0, 1, 2, 3])
    ax[j,k].set_yticks([0, 1, 2, 3])
    ax[j,k].grid()
    ax[j,k].set_aspect("equal")

    if i == 7:
        plt.legend(loc="lower right")

    ax[j,k].text(0.08, 0.92, char, transform = ax[j,k].transAxes, ha="center", va="center", fontweight="bold", 
            zorder=5, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))
     
fig.supxlabel("Normal condition BLH quantiles in km")
fig.supylabel("Hot day BLH quantiles in km")

plt.savefig('./figs/prudence_qq_10to22.pdf', bbox_inches='tight', format='pdf')
plt.show()

# Station Counts

In [ ]:
from matplotlib.colors import BoundaryNorm

In [ ]:
_ = [pd.read_csv(f"./data/obs/blhs_era5_obs_vt_{YYYY}.csv", index_col=0, parse_dates=["DateTime"]) for YYYY in range(2010,2023)]
df_obs = pd.concat(_, ignore_index=True)
df_obs = df_obs[df_obs.ID != 0.] #for some reason ID 0 does weird things

id_counts = df_obs.value_counts(["ID"]).reset_index(name="count") 
lons = np.array([np.mean(df_obs[df_obs["ID"] == id]["lon"], axis=0) for id in id_counts["ID"]])
lats = np.array([np.mean(df_obs[df_obs["ID"] == id]["lat"], axis=0) for id in id_counts["ID"]])

In [ ]:
fig = plt.figure()
ax = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

ax.set_extent([-14, 38, 35, 72], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.COASTLINE)

gl1 = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, alpha=0.15, zorder=10, color="black")
gl1.top_labels = False
gl1.right_labels = False

# Define bin edges and labels
bins = [100, 300, 600, 900, 1200, 1500, 1800, 2100, 2400, 2700, 3000]
labels = ['100', '300', '600', '900', '1200', '1500', '1800', '2100', '2400', '2700', '>3000']

# Create custom colormap and norm
cmap = cm.viridis
norm = BoundaryNorm(bins, cmap.N)

scatter = ax.scatter(lons, lats, c=id_counts["count"],
    cmap=cmap, norm=norm, s=90,  edgecolor="k"
)

cbar = plt.colorbar(scatter, ticks=bins, ax=ax, fraction=0.033, pad=0.04, label="Number of observations")
cbar.ax.set_yticklabels(labels)

# Add rectangles and labels for PRUDENCE regions
for key, region in prudence_regions.items():
    rect = patches.Rectangle(
        (region["lon"].start, region["lat"].start),
        region["lon"].stop - region["lon"].start,
        region["lat"].stop - region["lat"].start,
        linewidth=1, edgecolor='black', facecolor='none', zorder=-1
    )
    ax.add_patch(rect)
       
    ax.text(region["label_pos"][0], region["label_pos"][1], key, 
            ha='center', va='center', fontweight='bold', zorder=1,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.savefig('./figs/station_counts.pdf', bbox_inches='tight', format='pdf')
plt.show()

# BLH Trends

In [ ]:
# Grid info:
rea_static = xr.open_dataset("./data/rea_static_rot.nc")
rea6_lsm = rea_static["LSM"].coarsen(dim={'rlat': 3, 'rlon': 3}, boundary='trim').max(skipna=True)
rea6_lsm = rea6_lsm.isel(rlat=slice(49,None), rlon=slice(29,None))

era5_lsm = xr.open_dataset("./data/era5_lsm.nc").squeeze()["lsm"]
era5_lsm["longitude"] = xr.where(era5_lsm["longitude"] > 180., era5_lsm["longitude"] - 360, era5_lsm["longitude"])


# BLHs:
blh_rea6 = xr.open_dataset("./data/ifs_blh_dmax_rea6.nc")["blh"].isel(rlat=slice(49,None),rlon=slice(29,None)).where(rea6_lsm > 0.5)

blh_era5 = xr.open_dataset("./data/blh_dmax_era5.nc")["blh"].where(era5_lsm > 0.5)
blh_era5["valid_time"] = blh_era5["valid_time"].dt.floor("D")

In [ ]:
coeffs_75_rea6 = {}
lengths_rea6 = {}

coeffs_75_era5 = {}
lengths_era5 = {}

for key in prudence_regions.keys():
    print(f"Processing: {prudence_regions[key]["name"]}")


    ### REA6 ###
    coeffs_region = []
    lengths_region = []

    # Load HW definition for region:
    EPI = xr.open_dataset(f"./data/EPI/REA6/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    EPI["time"] = EPI["time"].dt.floor("D")
    q_epi_region = np.quantile(EPI, epi_threshold)


    # Region Mask
    mask_lat = (blh_rea6["lat"] >= (prudence_regions[key]["lat"]).start) & (blh_rea6["lat"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_rea6["lon"] >= (prudence_regions[key]["lon"]).start) & (blh_rea6["lon"] <= (prudence_regions[key]["lon"]).stop)
    blh_rea6_region = blh_rea6.where(mask_lat & mask_lon)
    
    filled_dts = pd.date_range(start=EPI["time"].min().values, end=EPI["time"].max().values, freq="D")
    filled_EPI = (EPI > q_epi_region).reindex(time=filled_dts, fill_value=False)
    events = find_events(filled_EPI)
    
    for s, e, l in events:      
        ds = blh_rea6_region.sel(time=slice(s, e))

        if ds.sizes["time"] < 5: #filter out events shorter than 3 days
            continue

        qs = ds.quantile(0.75, dim=["rlat", "rlon"]).T
        slope, intercept = np.polyfit(range(qs.sizes["time"]), qs, 1)
        coeffs_region.append(slope)
        lengths_region.append(l)

    coeffs_75_rea6[key] = np.array(coeffs_region)
    lengths_rea6[key] = np.array(lengths_region)

    print("    -> REA6 done")
    

    ### ERA5 ###
    coeffs_region = []
    lengths_region = []

    # Load HW definition for region:
    EPI = xr.open_dataset(f"./data/EPI/ERA5/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    EPI["time"] = EPI["time"].dt.floor("D")
    q_epi_region = np.quantile(EPI, epi_threshold)

    # Region Mask
    mask_lat = (blh_era5["latitude"] >= (prudence_regions[key]["lat"]).start) & (blh_era5["latitude"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_era5["longitude"] >= (prudence_regions[key]["lon"]).start) & (blh_era5["longitude"] <= (prudence_regions[key]["lon"]).stop)
    blh_era5_region = blh_era5.where(mask_lat & mask_lon)

    filled_dts = pd.date_range(start=EPI["time"].min().values, end=EPI["time"].max().values, freq="D")
    filled_EPI = (EPI > q_epi_region).reindex(time=filled_dts, fill_value=False)
    events = find_events(filled_EPI)
    
    for s, e, l in events:      
        ds = blh_era5_region.sel(valid_time=slice(s, e))

        if ds.sizes["valid_time"] < 3: #filter out events shorter than 3 days
            continue
        
        qs = ds.quantile(0.75, dim=["latitude", "longitude"]).T
        slope, intercept = np.polyfit(range(qs.sizes["valid_time"]), qs, 1)
        coeffs_region.append(slope)
        lengths_region.append(l)

    coeffs_75_era5[key] = np.array(coeffs_region)
    lengths_era5[key] = np.array(lengths_region) 

    print("    -> ERA5 done")
    print()

In [ ]:
fig, axs = plt.subplots(1, 2, tight_layout=False, figsize=(8,4), sharex=True, sharey=True)

for i, key in enumerate(coeffs_75_rea6.keys()):
    vals = coeffs_75_rea6[key]
    scatter1 = axs[0].scatter(np.repeat(i, len(vals)), vals, marker="+")
    axs[0].text(i, np.nanmax(coeffs_75_rea6[key]) + 60, str(np.nansum(coeffs_75_rea6[key] > 0.)), ha="center")
    axs[0].text(i, min(-100, np.nanmin(coeffs_75_rea6[key]) - 110), str(np.nansum(coeffs_75_rea6[key] < 0.)), ha="center")

axs[0].axhline(color="black")
axs[0].set_xticks(range(8), prudence_regions.keys())
axs[0].set(title="COSMO-REA6", ylabel="75th pct. BLH growth rate in m/day")
axs[0].text(0.055, 0.95, "a)", transform = axs[0].transAxes, ha="center", va="center", fontweight="bold", 
            zorder=5, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

for i, key in enumerate(coeffs_75_era5.keys()):
    vals = coeffs_75_era5[key]
    scatter2 = axs[1].scatter(np.repeat(i, len(vals)), vals, marker="+")
    axs[1].text(i, np.nanmax(coeffs_75_era5[key]) + 60, str(np.nansum(coeffs_75_era5[key] > 0.)), ha="center")
    axs[1].text(i, np.nanmin(coeffs_75_era5[key]) - 110, str(np.nansum(coeffs_75_era5[key] < 0.)), ha="center")

axs[1].axhline(color="black")
axs[1].set_xticks(range(8), prudence_regions.keys())
axs[1].set(title="ERA5")
axs[1].text(0.055, 0.95, "b)", transform = axs[1].transAxes, ha="center", va="center", fontweight="bold", 
            zorder=5, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

plt.ylim(-1200, 1100)

plt.savefig('./figs/prudence_tendencies_75.pdf', bbox_inches='tight', format='pdf')
plt.show()

# Tracing 

In [ ]:
def mask_above_150hpa(da):
    """
    Keep only the lowest 150 hPa of each profile.
    Surface pressure is approximated from the highest non-NaN level.
    """
    levels = da["level"]  
    valid = da.notnull() #true where data is not NaN
    
    # Approx surface pressure: max level value where data is valid
    # Use `where` to ignore NaN-levels, then take max along level dim
    approx_sp = levels.where(valid).max(dim="level") 
    
    # Mask: keep only where level > (approx_sp - 150)
    mask = levels > (approx_sp - 150)
    
    return da.where(mask)

In [ ]:
# Grid info:
era5_lsm = xr.open_dataset("./data/era5_lsm.nc").squeeze()["lsm"]
era5_lsm["longitude"] = xr.where(era5_lsm["longitude"] > 180., era5_lsm["longitude"] - 360, era5_lsm["longitude"])

# BLHs:
blh_era5 = xr.open_dataset("./data/blh_dmax_era5.nc")["blh"].where(era5_lsm > 0.5)
blh_era5["valid_time"] = blh_era5["valid_time"].dt.floor("D")
blh_era5 = blh_era5.sel(valid_time=slice(np.datetime64("2010"), np.datetime64("2023")))

The following script takes one to two hours to execute:

In [ ]:
dfs_reg = {} # regional results over heatwaves

# Average normal condition results saved in:
df_nc = pd.DataFrame(data=np.full((8,7), np.nan), columns=["region", "horizontal", "vertical", "diabatic", "seasonal", "initial", "BLH"])
df_nc["region"] = list(prudence_regions.keys())
df_nc.set_index("region", inplace=True)

nc_dict = {} # keep full distributions for boxplots


for key in prudence_regions.keys():
    print(f"Processing {prudence_regions[key]["name"]}")
    regional_quantiles = []

    # Load HW definition for region:
    EPI = xr.open_dataset(f"./data/EPI/ERA5/{key}/ExtremalPatternIndex_TPDM.nc")["EPI"]
    EPI["time"] = EPI["time"].dt.floor("D")
    q_epi_region = np.quantile(EPI, epi_threshold)   

    # Regional fields:
    mask_lat = (blh_era5["latitude"] >= (prudence_regions[key]["lat"]).start) & (blh_era5["latitude"] <= (prudence_regions[key]["lat"]).stop)
    mask_lon = (blh_era5["longitude"] >= (prudence_regions[key]["lon"]).start) & (blh_era5["longitude"] <= (prudence_regions[key]["lon"]).stop)
    ds_blh_region = blh_era5.where(mask_lat & mask_lon & (era5_lsm > 0.5))

    ds_anom_region = xr.open_mfdataset([f"./data/tracing/temperature_diagnostics_{key}_{YYYY}.nc" for YYYY in range(2010,2023)]) 


    # Process individual HWs
    # Scan trough EPI for multi-day events:
    filled_dts = pd.date_range(start=EPI["time"].min().values, end=EPI["time"].max().values, freq="D")
    filled_EPI = (EPI > q_epi_region).reindex(time=filled_dts, fill_value=False)
    events = find_events(filled_EPI)
    
    # Loop through events:
    start_dates = []
    for s, e, l in events:   

        # Time selection:
        ds_blh = ds_blh_region.sel(valid_time=slice(s, e))
        ds_anom = ds_anom_region.sel(time=slice(s, e))

        if ds_blh.sizes["valid_time"] < 3: #skip short events
            continue

        # Compute the events quantiles:
        event_quantiles = np.full(6, np.nan)
        for i, var_key in enumerate(ds_anom.data_vars):
            event_quantiles[i] = mask_above_150hpa(ds_anom[var_key]).load().mean().item() # anomalies
        event_quantiles[-1] = ds_blh.quantile(pctl).item() # BLH
        
        regional_quantiles.append(event_quantiles)
        start_dates.append(ds_blh["valid_time"].values[0])
    
    # Organize data into pandas data frame:
    df = pd.DataFrame(data=regional_quantiles, columns=["horizontal", "vertical", "diabatic", "seasonal", "initial", "BLH"])
    df["total"] = df.iloc[:,:4].sum(axis=1)
    df["advection"] = df.loc[:,["horizontal", "vertical"]].sum(axis=1)
    df["start_date"] = start_dates

    dfs_reg[key] = df


    # Normal conditions
    # Select time:
    nc_days = EPI.where(EPI < q_epi_region, drop=True)["time"].values.astype("datetime64[D]")
    nc_days_anom = np.intersect1d(ds_anom_region["time"].values.astype("datetime64[D]"), nc_days)
    nc_days_blh = np.intersect1d(ds_blh_region["valid_time"].values.astype("datetime64[D]"), nc_days)

    ds_blh_region_nc = ds_blh_region.sel(valid_time=nc_days_blh, method="nearest")
    ds_anom_region_nc = ds_anom_region.sel(time=nc_days_anom)

    # Dump all normal condition values into dict:
    region_anoms = {}
    for i, var_key in enumerate(ds_anom.data_vars):
        region_anoms[var_key] = mask_above_150hpa(ds_anom_region_nc[var_key]).mean(dim=["level", "latitude", "longitude"]).values
    nc_dict[key] = region_anoms

    # Get averages to plot differences to the mean during HWs:
    nc_quantiles = np.full(6, np.nan)
    for i, var_key in enumerate(ds_anom.data_vars):
        nc_quantiles[i] = mask_above_150hpa(ds_anom_region_nc[var_key]).load().mean().item()
    nc_quantiles[-1] = ds_blh_region_nc.quantile(pctl).item() # BLH
    df_nc.loc[key] = nc_quantiles
    
df_nc["total"] = df_nc.iloc[:,:4].sum(axis=1)
df_nc["advection"] = df_nc.loc[:,["horizontal", "vertical"]].sum(axis=1)

## Significant relationships

In [ ]:
from scipy.stats import linregress

In [ ]:
terms = ["horizontal", "vertical", "advection", "diabatic", "initial", "total"]

for reg_key in ["IP", "FR", "MD", "AL", "BI", "SC", "ME", "EA"]:
    print(f"Region: {reg_key}")

    for midx, var_key in enumerate(terms):
        df = dfs_reg[reg_key]
        x, y = df[var_key] - df_nc.loc[reg_key][var_key], df["BLH"] - df_nc.loc[reg_key]["BLH"]
        result = linregress(x, y)

        if result.pvalue < 0.05:
            print(f"\t{var_key}: {result.slope:.01}")

## Plots

In [ ]:
fig, axs = plt.subplots(3, 2, figsize=(10,7))
fig.tight_layout(w_pad=0.5, h_pad=2)

terms = ["horizontal", "vertical", "diabatic", "initial", "seasonal", "total"]
labels = ["Horizontal", "Vertical", "Diabatic", "Initial", "Seasonal", "Total"]
chars = ["a)", "b)", "c)", "d)", "e)", "f)"]

for midx, var_key, label, char in zip(range(6), terms, labels, chars):
    i, j = np.unravel_index(midx, (3,2))
    ax = axs[i, j]
    ax.axhline(0, color="black", alpha=0.5, zorder=-5)

    if j == 0:
        ax.set(title=label, ylabel=r"$\overline{\theta'}$ in K")
    else:
        ax.set(title=label)
    
    for k, reg_key in enumerate(prudence_regions.keys()):
        if midx != 5:
            ax.boxplot(nc_dict[reg_key][var_key], positions=[k], tick_labels=[reg_key], sym="", notch=True)

            _ = dfs_reg[reg_key][var_key]
            ax.scatter(np.repeat(k, len(_)), _, marker="v")
        else:
            total = (nc_dict[reg_key]["horizontal"] 
                     + nc_dict[reg_key]["vertical"] 
                     + nc_dict[reg_key]["diabatic"] 
                     + nc_dict[reg_key]["initial"] 
                     + nc_dict[reg_key]["seasonal"])
            
            ax.boxplot(total, positions=[k], tick_labels=[reg_key], sym="", notch=True)

            _ = dfs_reg[reg_key]["total"]
            ax.scatter(np.repeat(k, len(_)), _, marker="v")

    ax.text(0.03, 0.93, char, transform=ax.transAxes, ha="center", va="center", fontweight="bold", 
            zorder=1, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

fig.savefig("./figs/overview_tracing_l150.pdf", bbox_inches='tight', format="pdf")
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(9,6), sharey=True)

terms = ["horizontal", "vertical", "advection", "diabatic", "initial", "total"]
labels = ["Horizontal", "Vertical", "Horizontal + Vertical", "Diabatic", "Initial", "Total"]
chars = ["a)", "b)", "c)", "d)", "e)", "f)"]

for midx, var_key, label, char in zip(range(6), terms, labels, chars):
    i, j = np.unravel_index(midx, (2,3))
    ax = axs[i, j]
    #ax.axvline(0, color="black", alpha=0.5)

    for reg_key, marker in zip(["IP", "FR", "MD", "AL"], ["v", "^", "<", ">"]):
        df = dfs_reg[reg_key]
        ax.scatter(df[var_key] - df_nc.loc[reg_key][var_key], df["BLH"] - df_nc.loc[reg_key]["BLH"], 
                   marker=marker, label=reg_key, color=prudence_regions[reg_key]["color"])

        #ax.scatter(df_nc.loc[reg_key][var_key], df_nc.loc[reg_key]["BLH"], marker=marker, color="black")

    ax.set(title=f"{label}")
    if i == 1:
        ax.set(xlabel=r"$\overline{\theta'}_\text{HW} - \overline{\theta'}_\text{NC}$" + " in K")
    if j == 0:
        ax.set(ylabel=r"$\text{BLH}_\text{HW} - \text{BLH}_\text{NC}$ (75th pct) in m")

    #for x, y, s in zip(dfs_reg[reg_key][var_key], df["BLH"], df["start_date"]):
    #    ax.text(x, y, s.year)
    ax.axvline(0, color="black", alpha=0.5, zorder=-1)
    ax.axhline(0, color="black", alpha=0.5, zorder=-1)

    ax.text(0.08, 0.94, char, transform = ax.transAxes, ha="center", va="center", fontweight="bold", 
            zorder=1, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

axs[-1,-1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('./figs/tracing_dists_1_l150.pdf', bbox_inches='tight', format='pdf')
plt.show()

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(9,6), sharey=True)

terms = ["horizontal", "vertical", "advection", "diabatic", "initial", "total"]
labels = ["Horizontal", "Vertical", "Horizontal + Vertical", "Diabatic", "Initial", "Total"]
chars = ["a)", "b)", "c)", "d)", "e)", "f)"]

for midx, var_key, label, char in zip(range(6), terms, labels, chars):
    i, j = np.unravel_index(midx, (2,3))
    ax = axs[i, j]
    #ax.axvline(0, color="black", alpha=0.5)

    for reg_key, marker in zip(["BI", "SC", "ME", "EA"], ["v", "^", "<", ">"]):
        df = dfs_reg[reg_key]
        ax.scatter(df[var_key] - df_nc.loc[reg_key][var_key], df["BLH"] - df_nc.loc[reg_key]["BLH"], 
                   marker=marker, label=reg_key, color=prudence_regions[reg_key]["color"])

        #ax.scatter(df_nc.loc[reg_key][var_key], df_nc.loc[reg_key]["BLH"], marker=marker, color="black")

    ax.set(title=f"{label}")
    if i == 1:
        ax.set(xlabel=r"$\overline{\theta'}_\text{HW} - \overline{\theta'}_\text{NC}$" + " in K")
    if j == 0:
        ax.set(ylabel=r"$\text{BLH}_\text{HW} - \text{BLH}_\text{NC}$ (75th pct) in m")

    #for x, y, s in zip(dfs_reg[reg_key][var_key], df["BLH"], df["start_date"]):
    #    ax.text(x, y, s.year)
    ax.axvline(0, color="black", alpha=0.5, zorder=-1)
    ax.axhline(0, color="black", alpha=0.5, zorder=-1)

    ax.text(0.08, 0.94, char, transform = ax.transAxes, ha="center", va="center", fontweight="bold", 
            zorder=1, bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.7))

axs[-1,-1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('./figs/tracing_dists_2_l150.pdf', bbox_inches='tight', format='pdf')
plt.show()

# Case Studies

In [ ]:
# Station coordinates:
lat_t, lon_t = 48.777500, 2.002500 #Trappes
lat_v, lon_v = 51.660598, 39.200586 #Voronezh

# Station BLH timeseries:
# Reanalysis
blh_era5 = xr.open_dataset("./data/blh_dmax_era5.nc")["blh"]
blh_era5 = blh_era5.reindex(latitude=list(reversed(blh_era5["latitude"])))
blh_t = blh_era5.sel(latitude=lat_t, longitude=lon_t, method="nearest")
blh_v = blh_era5.sel(latitude=lat_v, longitude=lon_v, method="nearest")

# Observations
df_v_12 = pd.read_csv("./data/obs/blhs_voronezh_12.csv", index_col=0)

# Station HW definition:
hw_era5 = xr.open_dataset("./data/hw_gridded_era5.nc")["hw_ind"]
hw_t = hw_era5.sel(latitude=lat_t, longitude=lon_t, method="nearest")

tmax_v = xr.open_dataset("data/t2m_dmax_voronezh.grb").mean(dim=("latitude", "longitude"))["t2m"]
threshold_v = np.percentile(tmax_v, 95)
hw_v = tmax_v.where(tmax_v >= threshold_v)

## Time Series

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Trappes
axs[0].axhline(blh_t.quantile(0.9, skipna=True), color="grey")
axs[0].text(np.datetime64("2003-09-02"), blh_t.quantile(0.9, skipna=True)+100, "90pct", color="grey")

axs[0].plot(blh_t["valid_time"], blh_t, label="Closest Grid Point", color="tab:blue")
axs[0].plot(blh_era5["valid_time"], 
        blh_era5.sel(latitude=prudence_regions["FR"]["lat"], longitude=prudence_regions["FR"]["lon"]).mean(dim=("latitude", "longitude")),
        label="Regional Mean", linestyle="dashed", color="tab:blue")

blocks = find_events(np.isfinite(hw_t)) #hot day = finite value
axs[0].axvspan(hw_t["time"].sel(time=blocks[0][0], method="nearest").values, hw_t["time"].sel(time=blocks[0][1], method="nearest").values, 
               alpha=0.3, color="tab:red", label="Hot Days")
for start, end, l in blocks[1:]:
    axs[0].axvspan(hw_t["time"].sel(time=start, method="nearest").values - np.timedelta64(12, "h"), 
                   hw_t["time"].sel(time=end, method="nearest").values + np.timedelta64(12, "h"), alpha=0.3, color="tab:red")

#axs[0].legend(loc="upper left")
axs[0].set_xlim(np.datetime64("2003-06-01"), np.datetime64("2003-09-01"))
axs[0].set_ylim(0,5000)
axs[0].set(title="Trappes", ylabel="BLH in m")
axs[0].tick_params(axis="x", rotation=45)

axs[0].text(np.datetime64("2003-06-05"), 4750, "a)", 
        ha='center', va='center', fontweight='bold', zorder=1,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

# Voronezh
axs[1].axhline(blh_v.quantile(0.9, skipna=True), color="grey")
axs[1].text(np.datetime64("2010-09-02"), blh_v.quantile(0.9, skipna=True)+100, "90pct", color="grey")

axs[1].plot(blh_v["valid_time"], blh_v, label="Reanalysis")
axs[1].scatter(df_v_12["DateTime"], df_v_12["BLH"], marker="x", label="12 UTC Obs", color="black")

blocks = find_events(np.isfinite(hw_v))
axs[1].axvspan(hw_v["time"].sel(time=blocks[0][0], method="nearest").values, hw_v["time"].sel(time=blocks[0][1], method="nearest").values, 
               alpha=0.3, color="tab:red", label="HW")
for start, end, l in blocks[1:]:
    axs[1].axvspan(hw_v["time"].sel(time=start, method="nearest").values - np.timedelta64(12, "h"), 
                   hw_v["time"].sel(time=end, method="nearest").values + np.timedelta64(12, "h"), alpha=0.3, color="tab:red")

#axs[1].legend(loc="upper left")
axs[1].set_xlim(np.datetime64("2010-06-01"), np.datetime64("2010-09-01"))
axs[1].set_ylim(0,5000)
axs[1].set(title="Voronezh")
axs[1].tick_params(axis="x", rotation=45)

axs[1].text(np.datetime64("2010-06-05"), 4750, "b)", 
        ha='center', va='center', fontweight='bold', zorder=1,
        bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

plt.savefig('./figs/miralles_examples.pdf', bbox_inches='tight', format='pdf')
plt.show()

## Example Profiles

In [ ]:
import fdbck_tools as fb
from copy import copy

# Station coordinates:
lat_t, lon_t = 48.777500, 2.002500 #Trappes

In [ ]:
def find_closest_gp(lat, lon, lats, lons):
    """
    Returns the closes grid point of the `lats`/`lons` grid to `lat` and `lon` assuming
    `lats` is defined on every grid point.
    """
    i_min = ((lats - lat)**2 + (lons - lon)**2).argmin()
    if lats.ndim == 1:
        return i_min
    else:
        i_rlat, i_rlon = np.unravel_index(i_min, lats.shape)
        return i_rlat, i_rlon

In [ ]:
# Observations:
with open("./data/obs/obs_trappes_2015070112.pkl", "rb") as f:
    obs_dt = pickle.load(f)
    df_temp_obs = pickle.load(f)
    Z_0 = pickle.load(f)
    df_synop_obs = pickle.load(f)

df_temp = df_temp_obs[df_temp_obs.index <= df_synop_obs.index.item()] #cut extrapolated data

df_temp["Ri_b"] = fb.bulk_richardson_number(df_synop_obs, df_temp) #IFS
blh_obs_ifs = fb.compute_blh(df_temp, Z_0) + Z_0 #get the height

df_temp["Ri_b"] = fb.bri_alt(df_synop_obs, df_temp) #VT
blh_obs_vt = fb.compute_blh(df_temp, Z_0) + Z_0 #get the height


# ERA5 Model Equivalents:
with open("./data/obs/meera5_trappes_2015070112.pkl", "rb") as f:
    obs_dt = pickle.load(f)
    df_temp_meera5 = pickle.load(f)
    Z_0 = pickle.load(f)
    df_synop_meera5 = pickle.load(f)

df_temp = df_temp_meera5[df_temp_meera5.index <= df_synop_meera5.index.item()] # cut extrapolated data

df_temp["Ri_b"] = fb.bulk_richardson_number(df_synop_meera5, df_temp)
blh_meera5_ifs = fb.compute_blh(df_temp, Z_0) + Z_0

df_temp["Ri_b"] = fb.bri_alt(df_synop_meera5, df_temp)
blh_meera5_vt = fb.compute_blh(df_temp, Z_0) + Z_0


# ERA5 itself:
# For ERA5, heights of model levels need to be derived from surface pressure first.
ds_era5 = xr.open_dataset("./data/era5_trappes_tquv_ml_2015070112.grb").sel(latitude=lat_t, longitude=lon_t, method="nearest")
gpot_era5 = xr.open_dataset("./data/era5_trappes_z_ml_2015070112.grb").sel(latitude=lat_t, longitude=lon_t, method="nearest")["z"]
r_E = 6378e3
h_era5 = r_E * gpot_era5/9.80665 / (r_E - gpot_era5/9.80665)
# note that gpot is the gepotential, gpot/g geopotential height and h_era5 the geometric height

ml_def = pd.read_csv("./data/era5_ml_def.csv")
_ = xr.open_dataset("./data/era5_trappes_zlnsp_ml_2015070112.grb").sel(latitude=lat_t, longitude=lon_t, method="nearest")
lnsp, z_sfc_era5 = _["lnsp"], _["z"]
p_half = ml_def["a [Pa]"] + ml_def["b"] * np.exp(lnsp.values)
p_era5 = (p_half.values[:-1] + p_half.values[1:])/2 / 100

# In contrast to REA6, I don't have the VT BLH estimates for ERA5, but I want them in this comparison.
# The simplest solution was to put ERA5 into the observational data structure, so I can use the feeback
# tools for this. 
df_temp_era5 = pd.DataFrame(index=p_era5 * 100, columns=df_temp.columns)
df_temp_era5["Z"] = gpot_era5/9.80665 # convert to gpm for the feedback tools
df_temp_era5["T"] = ds_era5["t"]
df_temp_era5["FF2"] = ds_era5["u"]**2 + ds_era5["v"]**2
df_temp_era5["Q"] = ds_era5["q"]

df_synop_era5 = copy(df_synop_obs)
df_synop_era5.index = [np.exp(lnsp).item()]
df_synop_era5["Z"] = z_sfc_era5.item()/9.80665

_ = xr.open_dataset("./data/era5_trappes_tdt_2m_2015070112.grb").sel(latitude=lat_t, longitude=lon_t, method="nearest")
df_synop_era5["T"] = _["t2m"].item()
df_synop_era5["RH"] = fb.magnus(_["d2m"] - 273.15) / fb.magnus(_["t2m"] - 273.15)
df_synop_era5["Q"] = fb.sh_from_rh(df_synop_era5["RH"].item(), _["t2m"], df_synop_era5.index.item()/100).item()

df_temp_era5["Ri_b"] = fb.bulk_richardson_number(df_synop_era5, df_temp_era5)
blh_era5_ifs = fb.compute_blh(df_temp_era5, df_synop_era5["Z"].item()) + df_synop_era5["Z"].item()

df_temp_era5["Ri_b"] = fb.bri_alt(df_synop_era5, df_temp_era5)
blh_era5_vt = fb.compute_blh(df_temp_era5, df_synop_era5["Z"].item()) + df_synop_era5["Z"].item()


# REA6 Model Equivalents:
with open("./data/obs/merea6_trappes_2015070112.pkl", "rb") as f:
    obs_dt = pickle.load(f)
    df_temp_merea6 = pickle.load(f)
    Z_0 = pickle.load(f)
    df_synop_merea6 = pickle.load(f)

df_temp = df_temp_merea6[df_temp_merea6.index <= df_synop_merea6.index.item()] # cut extrapolated data

df_temp["Ri_b"] = fb.bulk_richardson_number(df_synop_merea6, df_temp) #IFS
blh_merea6_ifs = fb.compute_blh(df_temp, Z_0) + Z_0

df_temp["Ri_b"] = fb.bri_alt(df_synop_merea6, df_temp) #VT
blh_merea6_vt = fb.compute_blh(df_temp, Z_0) + Z_0


# REA6 itself:
rea_static = xr.open_dataset("./data/rea_static_rot.nc")
i_rlat, i_rlon = find_closest_gp(lat_t, lon_t, rea_static["lat"], rea_static["lon"])
rea6_h0 = rea_static["h0"].isel(rlon=i_rlon, rlat=i_rlat).item()

ds_rea6 = xr.open_dataset("./data/rea6_trappes_tquvph_ml_2015070112.nc")
blh_rea6_ifs = xr.open_dataset("./data/REA6_BLH/2015/ifs_blh_rea6_072015.nc")
blh_rea6_ifs = blh_rea6_ifs.isel(rlat=i_rlat, rlon=i_rlon).sel(time=np.datetime64("2015-07-01T12"))["blh"].item() + rea6_h0

blh_rea6_vt = xr.open_dataset("./data/REA6_BLH/2015/vt_blh_rea6_072015.nc")
blh_rea6_vt = blh_rea6_vt.isel(rlat=i_rlat, rlon=i_rlon).sel(time=np.datetime64("2015-07-01T12"))["blh"].item() + rea6_h0


# Height masks for plotting:
h_mask_obs = df_temp_obs["Z"] <= 5000
h_mask_era5 = h_era5 <= 5000
h_mask_meera5 = df_temp_meera5["Z"] <= 5000
h_mask_rea6 = ds_rea6["h"] <= 5000
h_mask_merea6 = df_temp_merea6["Z"] <= 5000


# ML Estimate
#ml_blh = xr.open_dataset("./data/REA6_BLH/2015/ml_blh_rea6_072015.nc")["blh"].isel(rlon=i_rlon, rlat=i_rlat).sel(time=np.datetime64("2015-07-01T12")) + rea6_h0


# Air parcel method:
# Since it's just a single example, I manually looked for the closest indeces to the surface potential temperature and interpolate:
potT_rea6 = fb.pot_t(ds_rea6["t"], ds_rea6["p"])[h_mask_rea6] 
potTv_rea6 = potT_rea6 * (0.608 * ds_rea6["q"] + 1)

rea6_t2m = xr.open_dataset("./data/REA6_BLH/2015/T2m/T_2M.2D.201507.grb").isel(x=i_rlon, y=i_rlat).sel(time=np.datetime64("2015-07-01T12"))["t2m"]
rea6_ps = xr.open_dataset("./data/REA6_BLH/2015/PS/PS.2D.201507.grb").isel(x=i_rlon, y=i_rlat).sel(time=np.datetime64("2015-07-01T12"))["sp"]
rea6_q = xr.open_dataset("./data/REA6_BLH/2015/QV2m/QV_2M.2D.201507.grb").isel(x=i_rlon, y=i_rlat).sel(time=np.datetime64("2015-07-01T12"))["QV_2M"]
potT_sfc = fb.pot_t(rea6_t2m, rea6_ps/100).item()
potTv_sfc = potT_sfc * (0.608 * rea6_q + 1)

parcel_blh = np.interp(potTv_sfc, potTv_rea6.values[-12:-14:-1], ds_rea6["h"].values[-12:-14:-1])
equiv_buoyancy_pt = np.interp(parcel_blh, ds_rea6["h"].values[-12:-14:-1], potT_rea6.values[-12:-14:-1])


# Liu and Liang Refinement:
for i in range(1,22):
    cond1 = potT_sfc < potT_rea6.values[-i]
    cond2 = ((potT_rea6.values[-(i+1)] - potT_rea6.values[-i])/(ds_rea6["h"][-(i+1)] - ds_rea6["h"][-i]) * 1e3).item() >= 4.
    if cond1 and cond2:
        break
    
liuliang_blh = ds_rea6["h"][-i]

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(9,4), sharey=True)

# Potential Temperature
ax = axs[0]
ax.set(xlabel=r"$\theta$ in K", ylabel="Height above sea level in km")
ax.set_yticks([0, 1000, 2000, 3000, 4000, 5000])
ax.set_yticklabels([0, 1, 2, 3, 4, 5])

# Vertical Profiles:
ax.plot(fb.pot_t(ds_era5["t"], p_era5)[h_mask_era5], h_era5[h_mask_era5], ".--", color="tab:red", zorder=1, alpha=0.6)
ax.scatter(fb.pot_t(df_temp_meera5["T"], df_temp_meera5.index/100)[h_mask_meera5], df_temp_meera5[h_mask_meera5]["Z"], marker="x", color="tab:red", zorder=2)

ax.plot(fb.pot_t(ds_rea6["t"], ds_rea6["p"])[h_mask_rea6], ds_rea6["h"][h_mask_rea6], ".--", color="tab:blue", zorder=1, alpha=0.6)
ax.scatter(fb.pot_t(df_temp_merea6["T"], df_temp_merea6.index/100)[h_mask_merea6], df_temp_merea6[h_mask_merea6]["Z"], marker="x", color="tab:blue", zorder=2)

ax.scatter(fb.pot_t(df_temp_obs["T"], df_temp_obs.index/100)[h_mask_obs], df_temp_obs[h_mask_obs]["Z"], marker="x", color="black", zorder=3)

# Surface Stations:
ax.scatter(fb.pot_t(df_synop_meera5["T"], df_synop_meera5.index/100), df_synop_meera5["Z"], color="tab:red", marker="x")
ax.scatter(fb.pot_t(df_synop_merea6["T"], df_synop_merea6.index/100), df_synop_merea6["Z"], color="tab:blue", marker="x")
ax.scatter(fb.pot_t(df_synop_obs["T"], df_synop_obs.index/100), df_synop_obs["Z"], color="black", marker="x")


# Wind Speed
ax = axs[1]
ax.set(xlabel=r"$|\vec{v}|$ in m/s")

ax.plot(np.sqrt(ds_era5["u"]**2 + ds_era5["v"]**2)[h_mask_era5], h_era5[h_mask_era5], ".--", color="tab:red", zorder=1, alpha=0.6)
ax.scatter(np.sqrt(df_temp_meera5["FF2"][h_mask_meera5]), df_temp_meera5[h_mask_meera5]["Z"], marker="x", color="tab:red", zorder=2)

ax.plot(np.sqrt(ds_rea6["u"]**2 + ds_rea6["v"]**2)[h_mask_rea6], ds_rea6["h"][h_mask_rea6], ".--", color="tab:blue", zorder=1, alpha=0.6)
ax.scatter(np.sqrt(df_temp_merea6["FF2"][h_mask_merea6]), df_temp_merea6[h_mask_merea6]["Z"], marker="x", color="tab:blue", zorder=2)

ax.scatter(np.sqrt(df_temp_obs["FF2"][h_mask_obs]), df_temp_obs[h_mask_obs]["Z"], marker="x", color="black", zorder=3)
ax.set_xlim(right=11)

# Humidity
ax = axs[2]
ax.set(xlabel="q in g/kg")

# Vertical Profiles:
ax.plot(ds_era5["q"][h_mask_era5] * 1e3, h_era5[h_mask_era5], ".--", color="tab:red", label="ERA5", zorder=1, alpha=0.6)
ax.scatter(df_temp_meera5["Q"][h_mask_meera5]* 1e3, df_temp_meera5[h_mask_meera5]["Z"], marker="x", color="tab:red", label="ERA5 (MEC)", zorder=2)

ax.plot(ds_rea6["q"][h_mask_rea6]* 1e3, ds_rea6["h"][h_mask_rea6], ".--", color="tab:blue", label="REA6", zorder=1, alpha=0.6)
ax.scatter(df_temp_merea6["Q"][h_mask_merea6]* 1e3, df_temp_merea6[h_mask_merea6]["Z"], marker="x", color="tab:blue", label="REA6 (MEC)", zorder=2)

ax.scatter(df_temp_obs["Q"][h_mask_obs]* 1e3, df_temp_obs[h_mask_obs]["Z"], marker="x", color="black", label="Obs", zorder=3)

# Surface Stations:
ax.scatter(df_synop_meera5["Q"]* 1e3, df_synop_meera5["Z"], color="tab:red", marker="x")
ax.scatter(df_synop_merea6["Q"]* 1e3, df_synop_merea6["Z"], color="tab:blue", marker="x")
ax.scatter(df_synop_obs["Q"]* 1e3, df_synop_obs["Z"], color="black", marker="x")


# BLH estimates:
ax = axs[1]
ax.text(3., 1000, "VT-BLH")

ax.axhline(blh_obs_vt, xmin=0., xmax=0.3, color="black", zorder=0) # Obs
ax.axhline(blh_meera5_vt, xmin=0., xmax=0.3, color="tab:red", zorder=0) # ERA5
ax.axhline(blh_era5_vt, xmin=0., xmax=0.3, color="tab:red", zorder=0, linestyle="dashed")
ax.axhline(blh_merea6_vt, xmin=0., xmax=0.3, color="tab:blue", zorder=0) # REA6
ax.axhline(blh_rea6_vt, xmin=0., xmax=0.3, color="tab:blue", zorder=0, linestyle="dashed")


ax = axs[2]
ax.text(9.5, 2500, "IFS-BLH")

ax.axhline(blh_obs_ifs, xmin=0.7, xmax=1.0, color="black", zorder=0) # Obs
ax.axhline(blh_meera5_ifs, xmin=0.7, xmax=1.0, color="tab:red", zorder=0) # ERA5
ax.axhline(blh_era5_ifs, xmin=0.7, xmax=1.0, color="tab:red", zorder=0, linestyle="dashed")
ax.axhline(blh_merea6_ifs, xmin=0.7, xmax=1.0, color="tab:blue", zorder=0) # REA6
ax.axhline(blh_rea6_ifs, xmin=0.7, xmax=1.0, color="tab:blue", zorder=0, linestyle="dashed")


# Parcel demo:
ax = axs[0]
ax.scatter([potT_sfc, equiv_buoyancy_pt], [rea6_h0, parcel_blh], marker="o", facecolor="white", edgecolor="black", zorder=3)
ax.plot([potT_sfc, equiv_buoyancy_pt], [rea6_h0, parcel_blh], color="black", zorder=2)
ax.axhline(parcel_blh, xmin=0.35, color="black", label="Parcel")
ax.text(314, parcel_blh+35, "Parcel")

# L&L update:
ax.axhline(liuliang_blh, xmin=0.43, color="black", label="Liu & Liang")
ax.text(314, liuliang_blh+35, "L&L")

# ML demo:
#axs[1].axhline(ml_blh.item(), xmin=0.525, xmax=1.0, color="black")
#axs[1].text(10, ml_blh+35, "ML")


# Ground level markers and subplot labels
for ax, c in zip(axs, ["a)", "b)", "c)"]):
    ax.axhline(rea6_h0, color="tab:grey")
    ax.text(0.1, 0.95, c, transform = ax.transAxes, 
            ha='center', va='center', fontweight='bold', zorder=1,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7))

axs[2].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

plt.ylim(0,5500)
plt.savefig('./figs/trappes_2015_profiles.pdf', bbox_inches='tight', format='pdf')
plt.show()